# Laboratório 6 - Estimação de Profundidade com Câmera Estéreo

**Disciplina:** ESZA019 - Visão Computacional (UFABC)

**Equipe 8 - "Sem Título"**

**Integrantes:**

- Kayky de Brito dos Santos
- André Marques da Silva
- Rafael de Souza Coelho

**Relatório completo** (fotos, capturas de tela e resultados em texto): <https://kaykyb.github.io/ufabc-cv/posts/lab6/>

---

Notebook que consolida, em uma única sequência de células, todos os programas do roteiro do Laboratório 6:

1. **Captura** dos pares de calibração;
2. **Calibração** estéreo e geração dos mapas de retificação;
3. **Sintonia** dos parâmetros do `StereoBM` por interface gráfica;
4. **Calibração** da conversão disparidade → profundidade (constante `M`);
5. **Prevenção de obstáculos** com medição de distância ao vivo.

Os IDs das câmeras foram fixados em `0` (esquerda) e `1` (direita), e todos os caminhos de arquivo são relativos à pasta do notebook (`laboratorios/lab6/`), definidos uma única vez na célula de configuração para que as células troquem arquivos entre si de forma consistente.

> **Créditos:** os cinco programas foram **adaptados** do repositório oficial da LearnOpenCV (Satya Mallick), <https://github.com/spmallick/learnopencv> (projetos `stereo-camera` e `Depth-Perception-Using-StereoCamera`). As adaptações da equipe foram: fixar os IDs das câmeras, unificar os caminhos dos arquivos de parâmetros entre as etapas e organizar tudo em um único notebook.

> **Observação:** as células de captura e das GUIs (`3`, `5`) abrem janelas do OpenCV e dependem das duas webcams conectadas; execute-as localmente. Pressione `ESC` na janela ativa para encerrar cada laço.

## Configuração compartilhada

Definida uma única vez e reutilizada por todas as células seguintes.

In [ ]:
import os
import time

import cv2
import numpy as np
import matplotlib.pyplot as plt

# IDs das câmeras fixados
CamL_id = 0  # câmera esquerda
CamR_id = 1  # câmera direita

# Diretório de dados relativo ao notebook (laboratorios/lab6/)
DATA_DIR = "data"
PATH_L = os.path.join(DATA_DIR, "stereoL")
PATH_R = os.path.join(DATA_DIR, "stereoR")

# Arquivos de parâmetros compartilhados entre as células
PARAMS_FILE = os.path.join(DATA_DIR, "params_py.xml")  # mapas de retificação (saída da calibração)
DEPTH_PARAMS_FILE = os.path.join(DATA_DIR, "depth_estmation_params_py.xml")  # parâmetros do StereoBM + M

# Tabuleiro de calibração (cantos internos)
CHESSBOARD = (9, 6)

# Garante que os diretórios de dados existam
os.makedirs(PATH_L, exist_ok=True)
os.makedirs(PATH_R, exist_ok=True)

print("Config OK | câmeras L/R:", CamL_id, CamR_id, "| dados em:", os.path.abspath(DATA_DIR))

## 1. Captura dos pares de calibração

Adaptado de `stereo-camera/capture_images.py`. Mostra as duas câmeras ao vivo com um contador regressivo; a cada ciclo, se o tabuleiro for detectado **simultaneamente** nas duas imagens, o par é salvo em `data/stereoL/` e `data/stereoR/`. Pressione `ESC` para encerrar.

In [ ]:
CamL = cv2.VideoCapture(CamL_id)
CamR = cv2.VideoCapture(CamR_id)

start = time.time()
T = 10  # segundos entre capturas
count = 0

while True:
    timer = T - int(time.time() - start)
    retR, frameR = CamR.read()
    retL, frameL = CamL.read()

    if not (retR and retL):
        CamL = cv2.VideoCapture(CamL_id)
        CamR = cv2.VideoCapture(CamR_id)
        continue

    img1_temp = frameL.copy()
    cv2.putText(img1_temp, "%r" % timer, (50, 50), 1, 5, (55, 0, 0), 5)
    cv2.imshow("imgR", frameR)
    cv2.imshow("imgL", img1_temp)

    grayR = cv2.cvtColor(frameR, cv2.COLOR_BGR2GRAY)
    grayL = cv2.cvtColor(frameL, cv2.COLOR_BGR2GRAY)

    # Detecta os cantos do tabuleiro
    retR, cornersR = cv2.findChessboardCorners(grayR, CHESSBOARD, None)
    retL, cornersL = cv2.findChessboardCorners(grayL, CHESSBOARD, None)

    # Se os cantos foram detectados nas duas imagens, salva o par
    if (retR is True) and (retL is True) and timer <= 0:
        count += 1
        cv2.imwrite(os.path.join(PATH_R, "img%d.png" % count), frameR)
        cv2.imwrite(os.path.join(PATH_L, "img%d.png" % count), frameL)
        print("Par salvo:", count)

    if timer <= 0:
        start = time.time()

    if cv2.waitKey(1) & 0xFF == 27:  # ESC encerra
        print("Fechando as câmeras!")
        break

CamR.release()
CamL.release()
cv2.destroyAllWindows()

## 2. Calibração estéreo e mapas de retificação

Adaptado de `stereo-camera/calibrate.py`. Calibra cada câmera individualmente, faz a calibração estéreo com intrínsecos fixos (`CALIB_FIX_INTRINSIC`), retifica e grava os mapas de retificação em `PARAMS_FILE` (`data/params_py.xml`), o arquivo consumido pelas células seguintes. O número de pares é detectado automaticamente a partir dos arquivos salvos na etapa 1.

In [ ]:
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# Pontos 3D do tabuleiro (z = 0)
objp = np.zeros((CHESSBOARD[0] * CHESSBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD[0], 0:CHESSBOARD[1]].T.reshape(-1, 2)

img_ptsL, img_ptsR, obj_pts = [], [], []

# Número de pares detectado automaticamente
num_imgs = len([f for f in os.listdir(PATH_L) if f.startswith("img") and f.endswith(".png")])
print("Pares encontrados:", num_imgs)

for i in range(1, num_imgs + 1):
    imgL = cv2.imread(os.path.join(PATH_L, "img%d.png" % i))
    imgR = cv2.imread(os.path.join(PATH_R, "img%d.png" % i))
    if imgL is None or imgR is None:
        continue

    imgL_gray = cv2.cvtColor(imgL, cv2.COLOR_BGR2GRAY)
    imgR_gray = cv2.cvtColor(imgR, cv2.COLOR_BGR2GRAY)

    retL, cornersL = cv2.findChessboardCorners(imgL_gray, CHESSBOARD, None)
    retR, cornersR = cv2.findChessboardCorners(imgR_gray, CHESSBOARD, None)

    if retR and retL:
        obj_pts.append(objp)
        cv2.cornerSubPix(imgR_gray, cornersR, (11, 11), (-1, -1), criteria)
        cv2.cornerSubPix(imgL_gray, cornersL, (11, 11), (-1, -1), criteria)
        img_ptsL.append(cornersL)
        img_ptsR.append(cornersR)

print("Pares válidos usados:", len(obj_pts))

# Calibração individual de cada câmera
retL, mtxL, distL, rvecsL, tvecsL = cv2.calibrateCamera(
    obj_pts, img_ptsL, imgL_gray.shape[::-1], None, None
)
hL, wL = imgL_gray.shape[:2]
new_mtxL, roiL = cv2.getOptimalNewCameraMatrix(mtxL, distL, (wL, hL), 1, (wL, hL))

retR, mtxR, distR, rvecsR, tvecsR = cv2.calibrateCamera(
    obj_pts, img_ptsR, imgR_gray.shape[::-1], None, None
)
hR, wR = imgR_gray.shape[:2]
new_mtxR, roiR = cv2.getOptimalNewCameraMatrix(mtxR, distR, (wR, hR), 1, (wR, hR))

# Calibração estéreo mantendo os intrínsecos fixos
flags = cv2.CALIB_FIX_INTRINSIC
criteria_stereo = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
retS, new_mtxL, distL, new_mtxR, distR, Rot, Trns, Emat, Fmat = cv2.stereoCalibrate(
    obj_pts, img_ptsL, img_ptsR, new_mtxL, distL, new_mtxR, distR,
    imgL_gray.shape[::-1], criteria_stereo, flags,
)
print("RMS da calibração estéreo:", retS)

# Retificação estéreo
rectify_scale = 1  # 0 = corta a imagem, 1 = mantém a imagem inteira
rect_l, rect_r, proj_mat_l, proj_mat_r, Q, roiL, roiR = cv2.stereoRectify(
    new_mtxL, distL, new_mtxR, distR, imgL_gray.shape[::-1],
    Rot, Trns, rectify_scale, (0, 0),
)

# Mapas de retificação (pixel original -> pixel retificado/corrigido)
Left_Stereo_Map = cv2.initUndistortRectifyMap(
    new_mtxL, distL, rect_l, proj_mat_l, imgL_gray.shape[::-1], cv2.CV_16SC2
)
Right_Stereo_Map = cv2.initUndistortRectifyMap(
    new_mtxR, distR, rect_r, proj_mat_r, imgR_gray.shape[::-1], cv2.CV_16SC2
)

# Salva os mapas de retificação (consumidos pelas etapas 3, 4 e 5)
cv_file = cv2.FileStorage(PARAMS_FILE, cv2.FILE_STORAGE_WRITE)
cv_file.write("Left_Stereo_Map_x", Left_Stereo_Map[0])
cv_file.write("Left_Stereo_Map_y", Left_Stereo_Map[1])
cv_file.write("Right_Stereo_Map_x", Right_Stereo_Map[0])
cv_file.write("Right_Stereo_Map_y", Right_Stereo_Map[1])
cv_file.release()
print("Mapas de retificação salvos em:", PARAMS_FILE)

## 3. Sintonia dos parâmetros do StereoBM (GUI)

Adaptado de `Depth-Perception-Using-StereoCamera/disparity_params_gui.py`. Carrega os mapas de retificação de `PARAMS_FILE`, exibe o mapa de disparidade ao vivo e permite ajustar os parâmetros do `StereoBM` pelos *trackbars*. Ao pressionar `ESC`, grava os parâmetros escolhidos em `DEPTH_PARAMS_FILE` (`data/depth_estmation_params_py.xml`).

In [ ]:
CamL = cv2.VideoCapture(CamL_id)
CamR = cv2.VideoCapture(CamR_id)

# Lê os mapas de retificação gerados na etapa 2
cv_file = cv2.FileStorage(PARAMS_FILE, cv2.FILE_STORAGE_READ)
Left_Stereo_Map_x = cv_file.getNode("Left_Stereo_Map_x").mat()
Left_Stereo_Map_y = cv_file.getNode("Left_Stereo_Map_y").mat()
Right_Stereo_Map_x = cv_file.getNode("Right_Stereo_Map_x").mat()
Right_Stereo_Map_y = cv_file.getNode("Right_Stereo_Map_y").mat()
cv_file.release()


def nothing(x):
    pass


cv2.namedWindow("disp", cv2.WINDOW_NORMAL)
cv2.resizeWindow("disp", 600, 600)

cv2.createTrackbar("numDisparities", "disp", 1, 17, nothing)
cv2.createTrackbar("blockSize", "disp", 5, 50, nothing)
cv2.createTrackbar("preFilterType", "disp", 1, 1, nothing)
cv2.createTrackbar("preFilterSize", "disp", 2, 25, nothing)
cv2.createTrackbar("preFilterCap", "disp", 5, 62, nothing)
cv2.createTrackbar("textureThreshold", "disp", 10, 100, nothing)
cv2.createTrackbar("uniquenessRatio", "disp", 15, 100, nothing)
cv2.createTrackbar("speckleRange", "disp", 0, 100, nothing)
cv2.createTrackbar("speckleWindowSize", "disp", 3, 25, nothing)
cv2.createTrackbar("disp12MaxDiff", "disp", 5, 25, nothing)
cv2.createTrackbar("minDisparity", "disp", 5, 25, nothing)

stereo = cv2.StereoBM_create()

while True:
    retL, imgL = CamL.read()
    retR, imgR = CamR.read()

    if retL and retR:
        imgR_gray = cv2.cvtColor(imgR, cv2.COLOR_BGR2GRAY)
        imgL_gray = cv2.cvtColor(imgL, cv2.COLOR_BGR2GRAY)

        # Retificação dos dois quadros
        Left_nice = cv2.remap(imgL_gray, Left_Stereo_Map_x, Left_Stereo_Map_y,
                              cv2.INTER_LANCZOS4, cv2.BORDER_CONSTANT, 0)
        Right_nice = cv2.remap(imgR_gray, Right_Stereo_Map_x, Right_Stereo_Map_y,
                               cv2.INTER_LANCZOS4, cv2.BORDER_CONSTANT, 0)

        # Lê os parâmetros dos trackbars
        numDisparities = cv2.getTrackbarPos("numDisparities", "disp") * 16
        blockSize = cv2.getTrackbarPos("blockSize", "disp") * 2 + 5
        preFilterType = cv2.getTrackbarPos("preFilterType", "disp")
        preFilterSize = cv2.getTrackbarPos("preFilterSize", "disp") * 2 + 5
        preFilterCap = cv2.getTrackbarPos("preFilterCap", "disp")
        textureThreshold = cv2.getTrackbarPos("textureThreshold", "disp")
        uniquenessRatio = cv2.getTrackbarPos("uniquenessRatio", "disp")
        speckleRange = cv2.getTrackbarPos("speckleRange", "disp")
        speckleWindowSize = cv2.getTrackbarPos("speckleWindowSize", "disp") * 2
        disp12MaxDiff = cv2.getTrackbarPos("disp12MaxDiff", "disp")
        minDisparity = cv2.getTrackbarPos("minDisparity", "disp")

        # Aplica os parâmetros
        stereo.setNumDisparities(numDisparities)
        stereo.setBlockSize(blockSize)
        stereo.setPreFilterType(preFilterType)
        stereo.setPreFilterSize(preFilterSize)
        stereo.setPreFilterCap(preFilterCap)
        stereo.setTextureThreshold(textureThreshold)
        stereo.setUniquenessRatio(uniquenessRatio)
        stereo.setSpeckleRange(speckleRange)
        stereo.setSpeckleWindowSize(speckleWindowSize)
        stereo.setDisp12MaxDiff(disp12MaxDiff)
        stereo.setMinDisparity(minDisparity)

        # Calcula e normaliza a disparidade
        disparity = stereo.compute(Left_nice, Right_nice)
        disparity = disparity.astype(np.float32)
        disparity = (disparity / 16.0 - minDisparity) / numDisparities

        cv2.imshow("disp", disparity)

        if cv2.waitKey(1) == 27:  # ESC encerra
            break
    else:
        CamL = cv2.VideoCapture(CamL_id)
        CamR = cv2.VideoCapture(CamR_id)

print("Salvando parâmetros de estimação de profundidade ......")
cv_file = cv2.FileStorage(DEPTH_PARAMS_FILE, cv2.FILE_STORAGE_WRITE)
cv_file.write("numDisparities", numDisparities)
cv_file.write("blockSize", blockSize)
cv_file.write("preFilterType", preFilterType)
cv_file.write("preFilterSize", preFilterSize)
cv_file.write("preFilterCap", preFilterCap)
cv_file.write("textureThreshold", textureThreshold)
cv_file.write("uniquenessRatio", uniquenessRatio)
cv_file.write("speckleRange", speckleRange)
cv_file.write("speckleWindowSize", speckleWindowSize)
cv_file.write("disp12MaxDiff", disp12MaxDiff)
cv_file.write("minDisparity", minDisparity)
cv_file.write("M", 39.075)  # valor inicial; recalculado na etapa 4
cv_file.release()

CamL.release()
CamR.release()
cv2.destroyAllWindows()

## 4. Calibração disparidade → profundidade (constante `M`)

Adaptado de `Depth-Perception-Using-StereoCamera/disparity2depth_calib.py`. Mantendo o alvo em distâncias conhecidas (de `max_dist` a `min_dist`, de `sample_delta` em `sample_delta` cm), dê **duplo clique** no alvo na janela `disp` para registrar a disparidade a cada distância. Ao final, o notebook plota profundidade × disparidade, ajusta a constante `M` de `Z = M / d` por mínimos quadrados e a regrava em `DEPTH_PARAMS_FILE`.

In [ ]:
CamL = cv2.VideoCapture(CamL_id)
CamR = cv2.VideoCapture(CamR_id)

# Lê os mapas de retificação (etapa 2)
cv_file = cv2.FileStorage(PARAMS_FILE, cv2.FILE_STORAGE_READ)
Left_Stereo_Map_x = cv_file.getNode("Left_Stereo_Map_x").mat()
Left_Stereo_Map_y = cv_file.getNode("Left_Stereo_Map_y").mat()
Right_Stereo_Map_x = cv_file.getNode("Right_Stereo_Map_x").mat()
Right_Stereo_Map_y = cv_file.getNode("Right_Stereo_Map_y").mat()
cv_file.release()

# Distâncias de amostragem (em cm)
max_dist = 230  # distância máxima do alvo
min_dist = 50   # distância mínima mensurável
sample_delta = 40  # passo entre amostras

Z = max_dist
Value_pairs = []

# Lê os parâmetros do StereoBM (etapa 3)
cv_file = cv2.FileStorage(DEPTH_PARAMS_FILE, cv2.FILE_STORAGE_READ)
numDisparities = int(cv_file.getNode("numDisparities").real())
blockSize = int(cv_file.getNode("blockSize").real())
preFilterType = int(cv_file.getNode("preFilterType").real())
preFilterSize = int(cv_file.getNode("preFilterSize").real())
preFilterCap = int(cv_file.getNode("preFilterCap").real())
textureThreshold = int(cv_file.getNode("textureThreshold").real())
uniquenessRatio = int(cv_file.getNode("uniquenessRatio").real())
speckleRange = int(cv_file.getNode("speckleRange").real())
speckleWindowSize = int(cv_file.getNode("speckleWindowSize").real())
disp12MaxDiff = int(cv_file.getNode("disp12MaxDiff").real())
minDisparity = int(cv_file.getNode("minDisparity").real())
M = cv_file.getNode("M").real()
cv_file.release()


def mouse_click(event, x, y, flags, param):
    global Z
    if event == cv2.EVENT_LBUTTONDBLCLK:
        if disparity[y, x] > 0:
            Value_pairs.append([Z, disparity[y, x]])
            print("Distância: %r cm  | Disparidade: %r" % (Z, disparity[y, x]))
            Z -= sample_delta


cv2.namedWindow("disp", cv2.WINDOW_NORMAL)
cv2.resizeWindow("disp", 600, 600)
cv2.namedWindow("left image", cv2.WINDOW_NORMAL)
cv2.resizeWindow("left image", 600, 600)
cv2.setMouseCallback("disp", mouse_click)

stereo = cv2.StereoBM_create()

while True:
    retR, imgR = CamR.read()
    retL, imgL = CamL.read()

    if retL and retR:
        imgR_gray = cv2.cvtColor(imgR, cv2.COLOR_BGR2GRAY)
        imgL_gray = cv2.cvtColor(imgL, cv2.COLOR_BGR2GRAY)

        Left_nice = cv2.remap(imgL_gray, Left_Stereo_Map_x, Left_Stereo_Map_y,
                              cv2.INTER_LANCZOS4, cv2.BORDER_CONSTANT, 0)
        Right_nice = cv2.remap(imgR_gray, Right_Stereo_Map_x, Right_Stereo_Map_y,
                               cv2.INTER_LANCZOS4, cv2.BORDER_CONSTANT, 0)

        stereo.setNumDisparities(numDisparities)
        stereo.setBlockSize(blockSize)
        stereo.setPreFilterType(preFilterType)
        stereo.setPreFilterSize(preFilterSize)
        stereo.setPreFilterCap(preFilterCap)
        stereo.setTextureThreshold(textureThreshold)
        stereo.setUniquenessRatio(uniquenessRatio)
        stereo.setSpeckleRange(speckleRange)
        stereo.setSpeckleWindowSize(speckleWindowSize)
        stereo.setDisp12MaxDiff(disp12MaxDiff)
        stereo.setMinDisparity(minDisparity)

        disparity = stereo.compute(Left_nice, Right_nice)
        disparity = disparity.astype(np.float32)
        disparity = (disparity / 16.0 - minDisparity) / numDisparities

        cv2.imshow("disp", disparity)
        cv2.imshow("left image", imgL)

        if cv2.waitKey(1) == 27:  # ESC encerra
            break
        if Z < min_dist:
            break
    else:
        CamL = cv2.VideoCapture(CamL_id)
        CamR = cv2.VideoCapture(CamR_id)

CamL.release()
CamR.release()
cv2.destroyAllWindows()

# depth = M * (1 / disparity)  ->  ajuste por mínimos quadrados
value_pairs = np.array(Value_pairs)
z = value_pairs[:, 0]
disp = value_pairs[:, 1]
disp_inv = 1 / disp

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
ax1.plot(disp, z, "o-")
ax1.set(xlabel="Normalized disparity value", ylabel="Depth from camera (cm)",
        title="Relation between depth \n and corresponding disparity")
ax1.grid()
ax2.plot(disp_inv, z, "o-")
ax2.set(xlabel="Inverse disparity value (1/disp)", ylabel="Depth from camera (cm)",
        title="Relation between depth \n and corresponding inverse disparity")
ax2.grid()
plt.show()

# Resolve M por mínimos quadrados (decomposição QR)
coeff = np.vstack([disp_inv, np.ones(len(disp_inv))]).T
ret, sol = cv2.solve(coeff, z, flags=cv2.DECOMP_QR)
M = sol[0, 0]
C = sol[1, 0]
print("Valor de M =", M)

# Regrava os parâmetros com o M ajustado
cv_file = cv2.FileStorage(DEPTH_PARAMS_FILE, cv2.FILE_STORAGE_WRITE)
cv_file.write("numDisparities", numDisparities)
cv_file.write("blockSize", blockSize)
cv_file.write("preFilterType", preFilterType)
cv_file.write("preFilterSize", preFilterSize)
cv_file.write("preFilterCap", preFilterCap)
cv_file.write("textureThreshold", textureThreshold)
cv_file.write("uniquenessRatio", uniquenessRatio)
cv_file.write("speckleRange", speckleRange)
cv_file.write("speckleWindowSize", speckleWindowSize)
cv_file.write("disp12MaxDiff", disp12MaxDiff)
cv_file.write("minDisparity", minDisparity)
cv_file.write("M", M)
cv_file.release()

## 5. Prevenção de obstáculos e medição de distância

Adaptado de `Depth-Perception-Using-StereoCamera/obstacle_avoidance.py`. Lê os mapas de retificação e os parâmetros do StereoBM (com o `M` ajustado na etapa 4), calcula o mapa de profundidade ao vivo (`depth = M / disparity`), segmenta o objeto mais próximo e exibe `WARNING!`/`SAFE!` conforme a distância. Duplo clique na janela `disp` imprime a distância no ponto. Pressione `ESC` para encerrar.

In [ ]:
CamL = cv2.VideoCapture(CamL_id)
CamR = cv2.VideoCapture(CamR_id)

# Lê os mapas de retificação (etapa 2)
cv_file = cv2.FileStorage(PARAMS_FILE, cv2.FILE_STORAGE_READ)
Left_Stereo_Map_x = cv_file.getNode("Left_Stereo_Map_x").mat()
Left_Stereo_Map_y = cv_file.getNode("Left_Stereo_Map_y").mat()
Right_Stereo_Map_x = cv_file.getNode("Right_Stereo_Map_x").mat()
Right_Stereo_Map_y = cv_file.getNode("Right_Stereo_Map_y").mat()
cv_file.release()

disparity = None
depth_map = None

max_depth = 400   # distância máxima mensurável (cm)
min_depth = 50    # distância mínima mensurável (cm)
depth_thresh = 100.0  # limiar de distância SEGURA (cm)

# Lê os parâmetros do StereoBM + M (etapas 3 e 4)
cv_file = cv2.FileStorage(DEPTH_PARAMS_FILE, cv2.FILE_STORAGE_READ)
numDisparities = int(cv_file.getNode("numDisparities").real())
blockSize = int(cv_file.getNode("blockSize").real())
preFilterType = int(cv_file.getNode("preFilterType").real())
preFilterSize = int(cv_file.getNode("preFilterSize").real())
preFilterCap = int(cv_file.getNode("preFilterCap").real())
textureThreshold = int(cv_file.getNode("textureThreshold").real())
uniquenessRatio = int(cv_file.getNode("uniquenessRatio").real())
speckleRange = int(cv_file.getNode("speckleRange").real())
speckleWindowSize = int(cv_file.getNode("speckleWindowSize").real())
disp12MaxDiff = int(cv_file.getNode("disp12MaxDiff").real())
minDisparity = int(cv_file.getNode("minDisparity").real())
M = cv_file.getNode("M").real()
cv_file.release()


def mouse_click(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDBLCLK:
        print("Distância = %.2f cm" % depth_map[y, x])


cv2.namedWindow("disp", cv2.WINDOW_NORMAL)
cv2.resizeWindow("disp", 600, 600)
cv2.setMouseCallback("disp", mouse_click)

output_canvas = None
stereo = cv2.StereoBM_create()


def obstacle_avoid():
    # Máscara das regiões com profundidade abaixo do limiar seguro
    mask = cv2.inRange(depth_map, 10, depth_thresh)

    # Só considera obstáculos grandes (filtra ruído)
    if np.sum(mask) / 255.0 > 0.01 * mask.shape[0] * mask.shape[1]:
        contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        cnts = sorted(contours, key=cv2.contourArea, reverse=True)

        if cv2.contourArea(cnts[0]) > 0.01 * mask.shape[0] * mask.shape[1]:
            x, y, w, h = cv2.boundingRect(cnts[0])

            mask2 = np.zeros_like(mask)
            cv2.drawContours(mask2, cnts, 0, (255), -1)

            # Profundidade média do objeto mais próximo que o limiar seguro
            depth_mean, _ = cv2.meanStdDev(depth_map, mask=mask2)

            cv2.putText(output_canvas, "WARNING !", (x + 5, y - 40), 1, 2, (0, 0, 255), 2, 2)
            cv2.putText(output_canvas, "Object at", (x + 5, y), 1, 2, (100, 10, 25), 2, 2)
            cv2.putText(output_canvas, "%.2f cm" % depth_mean, (x + 5, y + 40), 1, 2, (100, 10, 25), 2, 2)
    else:
        cv2.putText(output_canvas, "SAFE!", (100, 100), 1, 3, (0, 255, 0), 2, 3)

    cv2.imshow("output_canvas", output_canvas)


while True:
    retR, imgR = CamR.read()
    retL, imgL = CamL.read()

    if retL and retR:
        output_canvas = imgL.copy()

        imgR_gray = cv2.cvtColor(imgR, cv2.COLOR_BGR2GRAY)
        imgL_gray = cv2.cvtColor(imgL, cv2.COLOR_BGR2GRAY)

        Left_nice = cv2.remap(imgL_gray, Left_Stereo_Map_x, Left_Stereo_Map_y,
                              cv2.INTER_LANCZOS4, cv2.BORDER_CONSTANT, 0)
        Right_nice = cv2.remap(imgR_gray, Right_Stereo_Map_x, Right_Stereo_Map_y,
                               cv2.INTER_LANCZOS4, cv2.BORDER_CONSTANT, 0)

        stereo.setNumDisparities(numDisparities)
        stereo.setBlockSize(blockSize)
        stereo.setPreFilterType(preFilterType)
        stereo.setPreFilterSize(preFilterSize)
        stereo.setPreFilterCap(preFilterCap)
        stereo.setTextureThreshold(textureThreshold)
        stereo.setUniquenessRatio(uniquenessRatio)
        stereo.setSpeckleRange(speckleRange)
        stereo.setSpeckleWindowSize(speckleWindowSize)
        stereo.setDisp12MaxDiff(disp12MaxDiff)
        stereo.setMinDisparity(minDisparity)

        disparity = stereo.compute(Left_nice, Right_nice)
        disparity = disparity.astype(np.float32)
        disparity = (disparity / 16.0 - minDisparity) / numDisparities

        depth_map = M / disparity  # profundidade em cm

        mask_temp = cv2.inRange(depth_map, min_depth, max_depth)
        depth_map = cv2.bitwise_and(depth_map, depth_map, mask=mask_temp)

        obstacle_avoid()

        cv2.resizeWindow("disp", 700, 700)
        cv2.imshow("disp", disparity)

        if cv2.waitKey(1) == 27:  # ESC encerra
            break
    else:
        CamL = cv2.VideoCapture(CamL_id)
        CamR = cv2.VideoCapture(CamR_id)

CamL.release()
CamR.release()
cv2.destroyAllWindows()